In [ ]:
import numpy as np
import pandas as pd
from itertools import product
import math
import random
from scipy.spatial import ConvexHull, QhullError
import os
import subprocess
import time
from numba import jit

## A.Read Files

### A.1.Read Input File

In [ ]:
# read job inp file
with open('AbaqusFileName.inp') as f:
    lines = f.readlines()
    
# Check for Parts
parts_index = [i for i, item in enumerate(lines) if '*Instance' in item]
if (len(parts_index) != 0):
    print(str(len(parts_index)) + ' Part(s) found:')
    print('')
    for i in parts_index:
        print (lines[i])
else:
    print('No Partfound')

In [ ]:
# Traget Part Upper Bone
lines_lower_bone  = lines[parts_index[4]:]

In [ ]:
# Traget Part Lower Bone
lines_upper_bone = lines[parts_index[3]:parts_index[4]]

In [ ]:
# Traget Part Scaffold
lines_scaffold = lines[parts_index[1]:parts_index[2]]

In [ ]:
# Traget Part Callus 
lines_callus = lines[parts_index[2]:parts_index[3]]

### A.2.Read Nodes

In [ ]:
## Nodes

# Callus
# get only nodes
callus_nodes_list = lines_callus[lines_callus.index('*Node\n')+1 : lines_callus.index('*Element, type=C3D10\n')]
# split and trim each line 
callus_nodes_list = [[float(i) for i in item.strip().split(',')] for item in callus_nodes_list]
# convert to numpy
callus_nodes = np.array(callus_nodes_list)
# check if offset
try: 
    callus_nodes_offset = np.array([float(i) for i in lines_callus[1].strip().split(',')])
    callus_nodes = callus_nodes + np.insert(callus_nodes_offset, 0, 0)
except ValueError:
    print("No offset in callus part")

# Scaffold
# get only nodes
scaffold_nodes_list = lines_scaffold[lines_scaffold.index('*Node\n')+1 : lines_scaffold.index('*Element, type=C3D10\n')]
# split and trim each line 
scaffold_nodes_list = [[float(i) for i in item.strip().split(',')] for item in scaffold_nodes_list]
# convert to numpy
scaffold_nodes = np.array(scaffold_nodes_list)
# check if offset
try: 
    scaffold_nodes_offset = np.array([float(i) for i in lines_scaffold[1].strip().split(',')])
    scaffold_nodes = scaffold_nodes + np.insert(scaffold_nodes_offset, 0, 0)
except ValueError:
    print("No offset in scaffold part")

# Upper Bone
# get only nodes
upper_bone_nodes_list = lines_upper_bone[lines_upper_bone.index('*Node\n')+1 : lines_upper_bone.index('*Element, type=C3D10\n')]
# split and trim each line 
upper_bone_nodes_list = [[float(i) for i in item.strip().split(',')] for item in upper_bone_nodes_list]
# convert to numpy
upper_bone_nodes = np.array(upper_bone_nodes_list)
# check if offset
try: 
    upper_bone_nodes_offset = np.array([float(i) for i in lines_upper_bone[1].strip().split(',')])
    upper_bone_nodes = upper_bone_nodes + np.insert(upper_bone_nodes_offset, 0, 0)
except ValueError:
    print("No offset in upper bone part")

# Lower Bone
# get only nodes
lower_bone_nodes_list = lines_lower_bone[lines_lower_bone.index('*Node\n')+1 : lines_lower_bone.index('*Element, type=C3D10\n')]
# split and trim each line 
lower_bone_nodes_list = [[float(i) for i in item.strip().split(',')] for item in lower_bone_nodes_list]
# convert to numpy
lower_bone_nodes = np.array(lower_bone_nodes_list)
# check if offset
try: 
    lower_bone_nodes_offset = np.array([float(i) for i in lines_lower_bone[1].strip().split(',')])
    lower_bone_nodes = lower_bone_nodes + np.insert(lower_bone_nodes_offset, 0, 0)
except ValueError:
    print("No offset in lower bone part")

In [ ]:
print("Callus")
# x
print("x:", min(callus_nodes[:, 1]), ",", max(callus_nodes[:, 1]))
# y
print("y:", min(callus_nodes[:, 2]), ",", max(callus_nodes[:, 2]))
# z
print("z:", min(callus_nodes[:, 3]), ",", max(callus_nodes[:, 3]))

In [ ]:
print("Scaffolld")
# x
print("x:", min(scaffold_nodes[:, 1]), ",", max(scaffold_nodes[:, 1]))
# y
print("y:", min(scaffold_nodes[:, 2]), ",", max(scaffold_nodes[:, 2]))
# z
print("z:", min(scaffold_nodes[:, 3]), ",", max(scaffold_nodes[:, 3]))

In [ ]:
print("Upper Bone")
# x
print("x:", min(upper_bone_nodes[:, 1]), ",", max(upper_bone_nodes[:, 1]))
# y
print("y:", min(upper_bone_nodes[:, 2]), ",", max(upper_bone_nodes[:, 2]))
# z
print("z:", min(upper_bone_nodes[:, 3]), ",", max(upper_bone_nodes[:, 3]))

In [ ]:
print("Lower Bone")
# x
print("x:", min(lower_bone_nodes[:, 1]), ",", max(lower_bone_nodes[:, 1]))
# y
print("y:", min(lower_bone_nodes[:, 2]), ",", max(lower_bone_nodes[:, 2]))
# z
print("z:", min(lower_bone_nodes[:, 3]), ",", max(lower_bone_nodes[:, 3]))

### A.3.Read Elements

In [ ]:
## Elements

# Callus
# get only elements
callus_elements_list = lines_callus[lines_callus.index('*Element, type=C3D10\n') + 1 : lines_callus.index('*Nset, nset=Set-1, generate\n')]
# split and trim each line 
callus_elements_list = [[float(i) for i in item.strip().split(',')] for item in callus_elements_list]
# convert to numpy
callus_elements = np.array(callus_elements_list)
callus_elements = np.column_stack((callus_elements, np.full((len(callus_elements),6), np.nan)))

# Scaffold
# get only elements
scaffold_elements_list = lines_scaffold[lines_scaffold.index('*Element, type=C3D10\n') + 1 : lines_scaffold.index('*Nset, nset=Set-1, generate\n')]
# split and trim each line 
scaffold_elements_list = [[float(i) for i in item.strip().split(',')] for item in scaffold_elements_list]
# convert to numpy
scaffold_elements = np.array(scaffold_elements_list)
scaffold_elements = np.column_stack((scaffold_elements, np.full((len(scaffold_elements),6), np.nan)))

# Upper Bone
# get only elements
upper_bone_elements_list = lines_upper_bone[lines_upper_bone.index('*Element, type=C3D10\n') + 1 : lines_upper_bone.index('*Nset, nset=Set-1, generate\n')]
# split and trim each line 
upper_bone_elements_list = [[float(i) for i in item.strip().split(',')] for item in upper_bone_elements_list]
# convert to numpy
upper_bone_elements = np.array(upper_bone_elements_list)
upper_bone_elements = np.column_stack((upper_bone_elements, np.full((len(upper_bone_elements),6), np.nan)))

# Lower Bone
# get only elements
lower_bone_elements_list = lines_lower_bone[lines_lower_bone.index('*Element, type=C3D10\n') + 1 : lines_lower_bone.index('*Nset, nset=Set-1, generate\n')]
# split and trim each line 
lower_bone_elements_list = [[float(i) for i in item.strip().split(',')] for item in lower_bone_elements_list]
# convert to numpy
lower_bone_elements = np.array(lower_bone_elements_list)
lower_bone_elements = np.column_stack((lower_bone_elements, np.full((len(lower_bone_elements),6), np.nan)))

## B.Grid Creation and Mapping

### B.1.Create Lattice

In [ ]:
# creating lattice
# to do: automate this part


lattice_dim_x = 6 # in mm
lattice_dim_y = 11 # in mm
lattice_dim_z = 6 # in mm

size_factor = 20.0

LATTICE_X = int(lattice_dim_x * size_factor) + 1
LATTICE_Y = int(lattice_dim_y * size_factor) + 1
LATTICE_Z = int(lattice_dim_z * size_factor) + 1

cells_cords = np.indices((LATTICE_X, LATTICE_Y, LATTICE_Z)).reshape(3, -1).T

cells_cords_mm = cells_cords / size_factor

# Manul Adjustment
cells_cords_mm[:, 0] -= 2.0001
cells_cords_mm[:, 1] -= 2.6001
cells_cords_mm[:, 2] -= 2.5001

age = np.full((len(cells_cords),), 666)
lattice = np.full((len(cells_cords),), -1)
in_elemnet = np.full((len(cells_cords),), -1)
stimulus = np.full((len(cells_cords),), -1)
in_part = np.full((len(cells_cords),), -1)

index = np.arange(0, len(cells_cords))

neighbours = np.column_stack((
    np.where((cells_cords[:, 0] < LATTICE_X - 1), cells_cords[:, 0] + 1, cells_cords[:, 0])
    , np.where((0 < cells_cords[:, 0]), cells_cords[:, 0] - 1, cells_cords[:, 0])
    , np.where((cells_cords[:, 1] < LATTICE_Y - 1), cells_cords[:, 1] + 1, cells_cords[:, 1])
    , np.where((0 < cells_cords[:, 1]), cells_cords[:, 1] - 1, cells_cords[:, 1])
    , np.where((cells_cords[:, 2] < LATTICE_Z - 1), cells_cords[:, 2] + 1, cells_cords[:, 2])
    , np.where((0 < cells_cords[:, 2]), cells_cords[:, 2] - 1, cells_cords[:, 2])
))

lattice_cells = np.column_stack((index, in_part, in_part, cells_cords, cells_cords_mm, age, lattice, in_elemnet, stimulus, neighbours))
map_arr_helper = np.arange(len(cells_cords)).reshape(LATTICE_X, LATTICE_Y, LATTICE_Z)

In [ ]:
# # if map_arr_helper is correct
# check = [map_arr_helper[int(i[1]), int(i[2]), int(i[3])] == i[0] for i in lattice_cells]
# lattice_cells[np.logical_not(check)]

In [ ]:
# column names
ind, orginal_part, new_part, cell_x, cell_y, cell_z, cell_mm_x, cell_mm_y, cell_mm_z, age, lattice, in_elemnet, stimulus, neighbour_p_x, neighbour_n_x, neighbour_p_y, neighbour_n_y, neighbour_p_z, neighbour_n_z = range(19)

### B.2.Upper Bone

In [ ]:
# mapping lattice to abaqus part Uppar Bone
# to do: slowest part of the code, make it faster

upper_bone_elements[:, 11:14] = [upper_bone_nodes[np.isin(upper_bone_nodes[:,0], i[1:11])].min(axis=0)[1:] for i in upper_bone_elements]
upper_bone_elements[:, 14:] = [upper_bone_nodes[np.isin(upper_bone_nodes[:,0], j[1:11])].max(axis=0)[1:] for j in upper_bone_elements]

upper_bone_hulls = [ConvexHull(upper_bone_nodes[np.isin(upper_bone_nodes[:,0], k[1:11])][:,1:]) for k in upper_bone_elements]
upper_bone_elements = np.column_stack((upper_bone_elements, upper_bone_hulls))

for l in upper_bone_elements:
    
    indices_min_max = (
        (lattice_cells[:, cell_mm_z] <= l[16]) & (lattice_cells[:, cell_mm_z] >= l[13]) &
        (lattice_cells[:, cell_mm_y] <= l[15]) & (lattice_cells[:, cell_mm_y] >= l[12]) &
        (lattice_cells[:, cell_mm_x] <= l[14]) & (lattice_cells[:, cell_mm_x] >= l[11])
    )
    
    cell_in_min_max = lattice_cells[indices_min_max]

    # Apply the condition to filter cell_in_min_max
    indices_filtered = np.all((np.dot(l[17].equations[:, :-1], cell_in_min_max[:, 6:9].T) + l[17].equations[:, -1].reshape(-1, 1)) < 0, axis=0)
    
    # Get the indices of lattice_cells that satisfy the filters
    indices_np_cells = np.where(indices_min_max)[0]

    # Replace the element in lattice_cells at the corresponding indices with filtered_cells
    lattice_cells[:, in_elemnet][indices_np_cells[indices_filtered]] = l[0]
    
    # in_part = 1 -- in_part = callus
    lattice_cells[:, orginal_part][indices_np_cells[indices_filtered]] = 111
    lattice_cells[:, new_part][indices_np_cells[indices_filtered]] = 111
    
    print(l[0])

### B.3.Lower Bone

In [ ]:
# mapping lattice to abaqus part Lower Bone
# to do: slowest part of the code, make it faster

lower_bone_elements[:, 11:14] = [lower_bone_nodes[np.isin(lower_bone_nodes[:,0], i[1:11])].min(axis=0)[1:] for i in lower_bone_elements]
lower_bone_elements[:, 14:] = [lower_bone_nodes[np.isin(lower_bone_nodes[:,0], j[1:11])].max(axis=0)[1:] for j in lower_bone_elements]

lower_bone_hulls = [ConvexHull(lower_bone_nodes[np.isin(lower_bone_nodes[:,0], k[1:11])][:,1:]) for k in lower_bone_elements]
lower_bone_elements = np.column_stack((lower_bone_elements, lower_bone_hulls))

for l in lower_bone_elements:
    
    indices_min_max = (
        (lattice_cells[:, cell_mm_z] <= l[16]) & (lattice_cells[:, cell_mm_z] >= l[13]) &
        (lattice_cells[:, cell_mm_y] <= l[15]) & (lattice_cells[:, cell_mm_y] >= l[12]) &
        (lattice_cells[:, cell_mm_x] <= l[14]) & (lattice_cells[:, cell_mm_x] >= l[11])
    )
    
    cell_in_min_max = lattice_cells[indices_min_max]

    # Apply the condition to filter cell_in_min_max
    indices_filtered = np.all((np.dot(l[17].equations[:, :-1], cell_in_min_max[:, 6:9].T) + l[17].equations[:, -1].reshape(-1, 1)) < 0, axis=0)
    
    # Get the indices of lattice_cells that satisfy the filters
    indices_np_cells = np.where(indices_min_max)[0]

    # Replace the element in lattice_cells at the corresponding indices with filtered_cells
    lattice_cells[:, in_elemnet][indices_np_cells[indices_filtered]] = l[0]
    
    # in_part = 1 -- in_part = callus
    lattice_cells[:, orginal_part][indices_np_cells[indices_filtered]] = 112
    lattice_cells[:, new_part][indices_np_cells[indices_filtered]] = 112
    
    print(l[0])

## B.4.Scaffold

In [2]:
# mapping lattice to abaqus part Scaffold
# to do: slowest part of the code, make it faster

scaffold_elements[:, 11:14] = [scaffold_nodes[np.isin(scaffold_nodes[:,0], i[1:11])].min(axis=0)[1:] for i in scaffold_elements]
scaffold_elements[:, 14:] = [scaffold_nodes[np.isin(scaffold_nodes[:,0], j[1:11])].max(axis=0)[1:] for j in scaffold_elements]

scaffold_hulls = [ConvexHull(scaffold_nodes[np.isin(scaffold_nodes[:,0], k[1:11])][:,1:]) for k in scaffold_elements]
scaffold_elements = np.column_stack((scaffold_elements, scaffold_hulls))

for l in scaffold_elements:
    
    indices_min_max = (
        (lattice_cells[:, cell_mm_z] <= l[16]) & (lattice_cells[:, cell_mm_z] >= l[13]) &
        (lattice_cells[:, cell_mm_y] <= l[15]) & (lattice_cells[:, cell_mm_y] >= l[12]) &
        (lattice_cells[:, cell_mm_x] <= l[14]) & (lattice_cells[:, cell_mm_x] >= l[11])
    )
    
    cell_in_min_max = lattice_cells[indices_min_max]

    # Apply the condition to filter cell_in_min_max
    indices_filtered = np.all((np.dot(l[17].equations[:, :-1], cell_in_min_max[:, 6:9].T) + l[17].equations[:, -1].reshape(-1, 1)) < 0, axis=0)
    
    # Get the indices of lattice_cells that satisfy the filters
    indices_np_cells = np.where(indices_min_max)[0]

    # Replace the element in lattice_cells at the corresponding indices with filtered_cells
    lattice_cells[:, in_elemnet][indices_np_cells[indices_filtered]] = l[0]
    
    # in_part = 1 -- in_part = callus
    lattice_cells[:, orginal_part][indices_np_cells[indices_filtered]] = 200
    lattice_cells[:, new_part][indices_np_cells[indices_filtered]] = 200
    
    print(l[0])

NameError: name 'scaffold_elements' is not defined

## B.5.Callus

In [ ]:
# mapping lattice to abaqus part Callus
# to do: slowest part of the code, make it faster

callus_elements[:, 11:14] = [callus_nodes[np.isin(callus_nodes[:,0], i[1:11])].min(axis=0)[1:] for i in callus_elements]
callus_elements[:, 14:] = [callus_nodes[np.isin(callus_nodes[:,0], j[1:11])].max(axis=0)[1:] for j in callus_elements]

callus_hulls = [ConvexHull(callus_nodes[np.isin(callus_nodes[:,0], k[1:11])][:,1:]) for k in callus_elements]
callus_elements = np.column_stack((callus_elements, callus_hulls))

for l in callus_elements:
    
    indices_min_max = (
        (lattice_cells[:, cell_mm_z] <= l[16]) & (lattice_cells[:, cell_mm_z] >= l[13]) &
        (lattice_cells[:, cell_mm_y] <= l[15]) & (lattice_cells[:, cell_mm_y] >= l[12]) &
        (lattice_cells[:, cell_mm_x] <= l[14]) & (lattice_cells[:, cell_mm_x] >= l[11])
    )
    
    cell_in_min_max = lattice_cells[indices_min_max]

    # Apply the condition to filter cell_in_min_max
    indices_filtered = np.all((np.dot(l[17].equations[:, :-1], cell_in_min_max[:, 6:9].T) + l[17].equations[:, -1].reshape(-1, 1)) < 0, axis=0)
    
    # Get the indices of lattice_cells that satisfy the filters
    indices_np_cells = np.where(indices_min_max)[0]

    # Replace the element in lattice_cells at the corresponding indices with filtered_cells
    lattice_cells[:, in_elemnet][indices_np_cells[indices_filtered]] = l[0]
    
    # in_part = 1 -- in_part = callus
    lattice_cells[:, orginal_part][indices_np_cells[indices_filtered]] = 100
    lattice_cells[:, new_part][indices_np_cells[indices_filtered]] = 100
    
    print(l[0])

## C.Biological Functions and Degradation Process

### C.1.Create Abaqus Job File

In [ ]:
def prepare_simulation_file(callus_number_elems, scaffold_number_elems):
    
    # Remove previous model file (from previous simulation)
    if os.path.exists("NewAbaqusFileName.txt"):
        os.remove("NewAbaqusFileName.txt")
    # Remove previous simulation file
    if os.path.exists("NewAbaqusFileName.inp"):
        os.remove("NewAbaqusFileName.inp")
    
    # Open files
#     myfile = open("cube.inp", "w")  # initial simulation file
#     mytxtfile = open("Cube_model.txt", "w")  # model file to be used in later iterations
    myfile = open("NewAbaqusFileName.inp", "w")  # initial simulation file
    mytxtfile = open("NewAbaqusFileName.txt", "w")  # model file to be used in later iterations
    infile = open("AbaqusFileName.inp", "r")

    # Read & copy job file until ELSET section
    lines = infile.readlines()
    section_index = [lines.index('** Section: Granulation_tissue\n'), lines.index('** Section: PLLA\n')]
    list_section = ['** Section: PLLA\n', '** Section: Granulation_tissue\n']
    
    # Read & copy job file until ELSET section
    for i in lines[:min(section_index)]:
        myfile.write(i)
        mytxtfile.write(i)

    if (section_index.index(min(section_index)) == 0):
        for elem in range(callus_number_elems):
            elemsetname = f"Elset_{elem+1}"
            materialname = f"material_callus_{elem+1}"
            mytxtfile.write(f"*Elset, elset={elemsetname}\n")
            mytxtfile.write(f"{elem+1}\n")
            mytxtfile.write(f"*Solid section, elset={elemsetname}, material={materialname}\n")
            mytxtfile.write("1.,\n")
    elif (section_index.index(min(section_index)) == 1):
        for elem in range(scaffold_number_elems):
            elemsetname = f"Elset_{elem+1}"
            materialname = f"material_scaffold_{elem+1}"
            mytxtfile.write(f"*Elset, elset={elemsetname}\n")
            mytxtfile.write(f"{elem+1}\n")
            mytxtfile.write(f"*Solid section, elset={elemsetname}, material={materialname}\n")
            mytxtfile.write("1.,\n")


    for j in lines[(min(section_index) + 1) : max(section_index)]:
        myfile.write(j)
        mytxtfile.write(j)

    if (section_index.index(max(section_index)) == 0):
        for elem in range(callus_number_elems):
            elemsetname = f"Elset_{elem+1}"
            materialname = f"material_callus_{elem+1}"
            mytxtfile.write(f"*Elset, elset={elemsetname}\n")
            mytxtfile.write(f"{elem+1}\n")
            mytxtfile.write(f"*Solid section, elset={elemsetname}, material={materialname}\n")
            mytxtfile.write("1.,\n")
    elif (section_index.index(max(section_index)) == 1):
        for elem in range(scaffold_number_elems):
            elemsetname = f"Elset_{elem+1}"
            materialname = f"material_scaffold_{elem+1}"
            mytxtfile.write(f"*Elset, elset={elemsetname}\n")
            mytxtfile.write(f"{elem+1}\n")
            mytxtfile.write(f"*Solid section, elset={elemsetname}, material={materialname}\n")
            mytxtfile.write("1.,\n")
    
    # Copy rest of the file
    myfile.write(lines[max(section_index) + 1])  # *Solid Section, elset=t, material=tt
    myfile.write(lines[max(section_index) + 2])  # ,
    myfile.write(lines[max(section_index) + 3])  # *End Part

    till_index = lines[(max(section_index) + 3):].index('1643., 0.3\n')
    for k in lines[(max(section_index) + 3):(max(section_index) + 3 + till_index + 1)]:
        myfile.write(k)
        mytxtfile.write(k)

    mytxtfile.close()  # close model text file

    # copy rest to initial job file
    final_index = lines[(max(section_index) + 4 + till_index):].index('** OUTPUT REQUESTS\n')
    for p in lines[(max(section_index) + 4 + till_index):(max(section_index) + 4 + till_index + final_index)]:
        myfile.write(p)
    
    # Write ending into job file (E and S output in dat file)
    myfile.write("**---------------- OUTPUT REQUESTS -------------------------------\n")
    myfile.write("** \n")
    myfile.write("*Restart, write, frequency=0\n")
    myfile.write("** \n")
    myfile.write("** FIELD OUTPUT: F-Output-1\n")
    myfile.write("**\n")
    myfile.write("*Output, field\n")
    myfile.write("*Element Output, directions=YES\n")
    myfile.write("E, S\n")
    myfile.write("**\n")
    myfile.write("**\n")
    myfile.write("*EL PRINT, FREQUENCY=100\n")
    myfile.write("EP, SP\n")
    myfile.write("**\n")
    myfile.write("**\n")
    myfile.write("** HISTORY OUTPUT: H-Output-1\n")
    myfile.write("**\n")
    myfile.write("**\n")
    myfile.write("*Output, history\n")
    myfile.write("*Energy Output\n")
    myfile.write("ALLAE,\n")
    myfile.write("*End Step\n")
    
    
    # Close files
    myfile.close()
    infile.close()

### C.2.Update Abaqus Job File

In [ ]:
def update_simulation_file(callus_number_elems, scaffold_number_elems, np_cells):
    # Remove previous simulation file
    if os.path.exists("NewAbaqusFileName.inp"):
        os.remove("NewAbaqusFileName.inp")
   
    mytxtfile = open("NewAbaqusFileName.txt", "r")
    myfile = open("NewAbaqusFileName.inp", "w")
   
    for line in mytxtfile:
        myfile.write(line)
       
    lattice_a = np_cells[:, lattice][np_cells[:, orginal_part] == 100]
    elem = np_cells[:, in_elemnet][np_cells[:, orginal_part] == 100]
    Y = np.where(
        lattice_a == 0, 0.2, np.where(
        lattice_a == 1, 0.2, np.where(
        lattice_a == 2, 8000, np.where(
        lattice_a == 4, 10, np.where(
        lattice_a == 5, 2, 0)))))
   
    P = np.where(
        lattice_a == 0, 0.167, np.where(
        lattice_a == 1, 0.167, np.where(
        lattice_a == 2, 0.3, np.where(
        lattice_a == 4, 0.167, np.where(
        lattice_a == 5, 0.167, 0)))))
    
    np_local_arr = np.column_stack((elem, lattice_a, Y, P, np.full((len(np_cells[np_cells[:, orginal_part] == 100]),), 1)))
    
    for i in range(callus_number_elems):
        s_agg = np.sum(np_local_arr[np_local_arr[:, 0] == (i + 1)], axis=0)
        if (sum(s_agg) == 0):
            Young_mod_elem = 0.2
            Poison_rat_elem = 0.167
        else:
            Young_mod_elem = s_agg[2] / s_agg[4]
            Poison_rat_elem = s_agg[3] / s_agg[4]
        materialname = f"material_callus_{i + 1}"
        myfile.write(f"*Material, name={materialname}\n")
        myfile.write("*Elastic\n")
        myfile.write(f"{Young_mod_elem}, {Poison_rat_elem}\n")
    
    lattice_b = np_cells[:, lattice][np_cells[:, orginal_part] == 200]
    elem_b = np_cells[:, in_elemnet][np_cells[:, orginal_part] == 200]
    Y_b = np.where(
        lattice_b == -1, 1643, np.where(
        lattice_b == 0, 0.2, np.where(
        lattice_b == 1, 0.2, np.where(
        lattice_b == 2, 8000, np.where(
        lattice_b == 4, 10, np.where(
        lattice_b == 5, 2, 0))))))
   
    P_b = np.where(
        lattice_b == -1, 0.3, np.where(
        lattice_b == 0, 0.167, np.where(
        lattice_b == 1, 0.167, np.where(
        lattice_b == 2, 0.3, np.where(
        lattice_b == 4, 0.167, np.where(
        lattice_b == 5, 0.167, 0))))))
    
    np_local_arr_b = np.column_stack((elem_b, lattice_b, Y_b, P_b, np.full((len(np_cells[np_cells[:, orginal_part] == 200]),), 1)))
    
    for j in range(scaffold_number_elems):
        s_agg_b = np.sum(np_local_arr_b[np_local_arr_b[:, 0] == (j + 1)], axis=0)
        if (sum(s_agg_b) == 0):
            Young_mod_elem_b = 0.2
            Poison_rat_elem_b = 0.167
        else:
            Young_mod_elem_b = s_agg_b[2] / s_agg_b[4]
            Poison_rat_elem_b = s_agg_b[3] / s_agg_b[4]
        materialname_b = f"material_scaffold_{j + 1}"
        myfile.write(f"*Material, name={materialname_b}\n")
        myfile.write("*Elastic\n")
        myfile.write(f"{Young_mod_elem_b}, {Poison_rat_elem_b}\n")
       
    
    # Write ending into job file
    myfile.write("**\n")
    myfile.write("** ----------------------------------------------------------------\n")
    myfile.write("**\n")
    myfile.write("** STEP: Step-1\n")
    myfile.write("** \n")
    myfile.write("*Step, name=Step-1, nlgeom=NO\n")
    myfile.write("*Static\n")
    myfile.write("0.1, 1., 1e-05, 1.,\n")
    myfile.write("** \n")
    myfile.write("**-------------- BOUNDARY CONDITIONS   ------------------------------------------\n")
    myfile.write("**\n")
    myfile.write("**\n")
    myfile.write("** Name: BC-1 Type: Displacement/Rotation\n")
    myfile.write("*Boundary\n")
    myfile.write("Set-42, 1, 1\n")
    myfile.write("Set-42, 2, 2\n")
    myfile.write("Set-42, 3, 3\n")
    myfile.write("Set-42, 4, 4\n")
    myfile.write("Set-42, 5, 5\n")
    myfile.write("Set-42, 6, 6\n")
    myfile.write("**\n")
    myfile.write("**--------------  Loads  ------------------------------------------\n")
    myfile.write("**\n")
    myfile.write("**\n")
    myfile.write("** Name: Load-2   Type: Pressure\n")
    myfile.write("*Dsload\n")
    myfile.write("Bone_up_load, P, 17.7\n")
    myfile.write("** Name: Load-4   Type: Concentrated force\n")
    myfile.write("*Cload\n")
    myfile.write("Set-51, 1, 0.\n")
    myfile.write("Set-51, 2, -5.7\n")
    myfile.write("**\n")
    myfile.write("**---------------- OUTPUT REQUESTS -------------------------------\n")
    myfile.write("** \n")
    myfile.write("*Restart, write, frequency=0\n")
    myfile.write("** \n")
    myfile.write("** FIELD OUTPUT: F-Output-1\n")
    myfile.write("**\n")
    myfile.write("*Output, field\n")
    myfile.write("*Element Output, directions=YES\n")
    myfile.write("E, S\n")
    myfile.write("**\n")
    myfile.write("**\n")
    myfile.write("*EL PRINT, FREQUENCY=100\n")
    myfile.write("EP, SP\n")
    myfile.write("**\n")
    myfile.write("**\n")
    myfile.write("** HISTORY OUTPUT: H-Output-1\n")
    myfile.write("**\n")
    myfile.write("**\n")
    myfile.write("*Output, history\n")
    myfile.write("*Energy Output\n")
    myfile.write("ALLAE,\n")
    myfile.write("*End Step\n")
    myfile.close()
   
    mytxtfile.close()  # close model text file

### Calculate Stimulus from Abaqus .dat file

In [ ]:
def calculate_stimulus():
    
    # read dat file
    with open('NewAbaqusFileName.dat') as f:
        lines = f.readlines()
        
    # Get Mapping of local to Global Elements
    lines_map_start = lines.index('LOCAL TO GLOBAL NODE AND ELEMENT MAPS\n')
    lines_ele_map_start = lines[lines.index('LOCAL TO GLOBAL NODE AND ELEMENT MAPS\n'):].index('                                                  element    element    \n')
    lines_ele_map_end = lines[lines.index('LOCAL TO GLOBAL NODE AND ELEMENT MAPS\n'):].index( '                            P R O B L E M   S I Z E\n')

    lines_ele_map = lines[(lines_map_start + lines_ele_map_start + 3):(lines_map_start + lines_ele_map_end)]

    lines_element_map = [item.strip().split() for item in lines_ele_map]
    element_map_df = pd.DataFrame(lines_element_map,  columns=["Instance", "local", "global"])
    element_map_df = element_map_df.dropna(how='any')
    element_map_df = element_map_df.astype({'local': 'int64', 'global': 'int64'})

    # Get required analysis output
    lines_sim_start = lines.index('    ELEMENT  PT FOOT-       EP1         EP2         EP3         SP1         SP2         SP3         \n') + 3
    lines_sim_end = lines.index('          THE ANALYSIS HAS BEEN COMPLETED\n') - 8
    lines_sim = lines[lines_sim_start:lines_sim_end]
    lines_sim = [[float(i) for i in item.strip().split()] for item in lines_sim]

    # convet to data frame
    sim_val_df = pd.DataFrame(lines_sim,  columns=["ELEMENT", "PT", "EP1", "EP2", "EP3", "SP1", "SP2", "SP3"])
    sim_val_df = sim_val_df.astype({'ELEMENT': 'int64', 'PT': 'int64'})

    sim_val_df["HP_read"] = - sim_val_df[["SP1", "SP2", "SP3"]].sum(axis = 1) / 3
    sim_proc_df = (sim_val_df.groupby("ELEMENT")[["EP1", "HP_read"]].sum() / 4).reset_index()

    cond1 = np.logical_and(abs(sim_proc_df['HP_read']) < 0.0016, abs(sim_proc_df['EP1']) < 0.0004)
    cond2 = np.logical_and(abs(sim_proc_df['HP_read']) < 0.15, abs(sim_proc_df['EP1']) < 0.05)
    cond3 = np.logical_and(sim_proc_df['HP_read'] < -0.15, abs(sim_proc_df['EP1']) < 0.15)

    sim_proc_df['cal_stimulus'] = np.where(cond1, 1, np.where(cond2, 2, np.where(cond3, 4, 5)))
    
    # add mapping
    sim_proc_df = sim_proc_df.merge(element_map_df, left_on='ELEMENT', right_on='global')
    
    # to do: below part needs to change with different names of parts
    sim_proc_df['in_part'] = np.where(sim_proc_df['Instance'] == 'CORTICAL_BONEN_UP-1', 111
                                      , np.where(sim_proc_df['Instance'] == 'CORTICAL_BONE_DOWN-1', 112
                                                 , np.where(sim_proc_df['Instance'] == 'CALLUSREG4-1', 100
                                                            , np.where(sim_proc_df['Instance'] == 'REG4-1', 200, -1))))
    
    stimulus_arr = sim_proc_df[['in_part', 'local', 'cal_stimulus']].to_numpy(copy=True)
    return stimulus_arr

### C.3.Neighbourhood Check

In [ ]:
def func_neigh_check(np_cells, ind_arr, index):
    x, y, z = np_cells[int(index), cell_x:cell_mm_x]
    x_p, x_m, y_p, y_m, z_p, z_m = np_cells[int(index), neighbour_p_x:]
        
    bb1 = ind_arr[int(x_p)][int(y)][int(z)]
    bb2 = ind_arr[int(x_m)][int(y)][int(z)]
    bb3 = ind_arr[int(x)][int(y_p)][int(z)]
    bb4 = ind_arr[int(x)][int(y_m)][int(z)]
    bb5 = ind_arr[int(x)][int(y)][int(z_p)]
    bb6 = ind_arr[int(x)][int(y)][int(z_m)]
    
    if ((np.isin(np_cells[bb1, stimulus], [2, 4, 5])) or (np_cells[bb1, new_part] == 200)):
        return True
    elif ((np.isin(np_cells[bb2, stimulus], [2, 4, 5])) or (np_cells[bb2, new_part] == 200)):
        return True
    elif ((np.isin(np_cells[bb3, stimulus], [2, 4, 5])) or (np_cells[bb3, new_part] == 200)):
        return True
    elif ((np.isin(np_cells[bb4, stimulus], [2, 4, 5])) or (np_cells[bb4, new_part] == 200)):
        return True
    elif ((np.isin(np_cells[bb5, stimulus], [2, 4, 5])) or (np_cells[bb5, new_part] == 200)):
        return True
    elif ((np.isin(np_cells[bb6, stimulus], [2, 4, 5])) or (np_cells[bb6, new_part] == 200)):
        return True
    
    return False

### C.4.Differentiation and Apoptosis

In [ ]:
def func_diff(np_cells, ind_arr, mature_age, it, list_msc_diff_coeff, list_apoptosis_coeff, out_arr, msc_max_age):
    
    n_check = [func_neigh_check(np_cells, ind_arr, i[0]) for i in np_cells]
    mature_mscs = (np_cells[:, new_part] == 100) & (np_cells[:, in_elemnet] != -1) & (np_cells[:, lattice] == 1) & (np_cells[:, age] >= mature_age) & (np_cells[:, age] != 666) & (np_cells[:, stimulus] != 1) & np.array(n_check)
    selected_indices = np.where(mature_mscs)[0]
    # for MSC
    if len(selected_indices) > 0:
        if it < ACTIVITY_MAX:
            choices = np.random.choice(selected_indices, size = math.floor(len(selected_indices)*list_msc_diff_coeff[0]), replace = False)
        else:
            choices = np.random.choice(selected_indices, size = math.floor(len(selected_indices)*list_msc_diff_coeff[1]), replace = False)
            
        diff_msc = len(np_cells[choices])
        np_cells[:, lattice][choices] = np_cells[:, stimulus][choices]
        np_cells[:, age][choices] = 1
    else:
        diff_msc = 0
        
    np_cells[:, age:in_elemnet][np_cells[:, in_elemnet] == -1] = 666, -1
    
    # Apoptosis
    
    # MSCs
    #dead_msc_count = len(np_cells[(np_cells[:, new_part] == 100) & (np_cells[:, lattice] == 1) & (np_cells[:, age] >= msc_max_age)])
    #np_cells[:, age:in_elemnet][(np_cells[:, new_part] == 100) & (np_cells[:, lattice] == 1) & (np_cells[:, age] >= msc_max_age)] = 0
       # fibroblasts
    m_msc = (np_cells[:, new_part] == 100) & (np_cells[:, in_elemnet] != -1) & (np_cells[:, lattice] == 1) & (np_cells[:, age] >= msc_max_age)
    m_msc_indices = np.where(m_msc)[0]
    if len(m_msc_indices) > 0:
        choices_msc = np.random.choice(m_msc_indices, size = math.floor(len(m_msc_indices)*list_apoptosis_coeff[0]), replace = False)
        np_cells[:, age:in_elemnet][choices_msc] = 0, 0
    
    # fibroblasts
    m_fib = (np_cells[:, new_part] == 100) & (np_cells[:, in_elemnet] != -1) & (np_cells[:, lattice] == 5) & (np.isin(np_cells[:, stimulus], [1, 2, 4]))
    m_fib_indices = np.where(m_fib)[0]
    if len(m_fib_indices) > 0:
        choices_fib = np.random.choice(m_fib_indices, size = math.floor(len(m_fib_indices)*list_apoptosis_coeff[1]), replace = False)
        np_cells[:, age:in_elemnet][choices_fib] = 0, 0
    
    
    # chondrocytes
    m_cho = (np_cells[:, new_part] == 100) & (np_cells[:, in_elemnet] != -1) & (np_cells[:, lattice] == 4) & (np.isin(np_cells[:, stimulus], [1, 2, 5]))
    m_cho_indices = np.where(m_cho)[0]
    if len(m_cho_indices) > 0:
        choices_cho = np.random.choice(m_cho_indices, size = math.floor(len(m_cho_indices)*list_apoptosis_coeff[2]), replace = False)
        np_cells[:, age:in_elemnet][choices_cho] = 0, 0
        
        
    # mature_osteoblasts
    m_ost = (np_cells[:, new_part] == 100) & (np_cells[:, in_elemnet] != -1) & (np_cells[:, lattice] == 2) & (np_cells[:, stimulus] == 1)
    m_ost_indices = np.where(m_ost)[0]
    if len(m_ost_indices) > 0:
        choices_ost = np.random.choice(m_ost_indices, size = math.floor(len(m_ost_indices)*list_apoptosis_coeff[3]), replace = False)
        np_cells[:, age:in_elemnet][choices_ost] = 0, 0
    
    out_arr.append([it + 1, diff_msc, math.floor(len(selected_indices)*list_apoptosis_coeff[0]), math.floor(len(m_fib_indices)*list_apoptosis_coeff[1]), math.floor(len(m_cho_indices)*list_apoptosis_coeff[2]), math.floor(len(m_ost_indices)*list_apoptosis_coeff[3])])
    print([it + 1, diff_msc, math.floor(len(selected_indices)*list_apoptosis_coeff[0]), math.floor(len(m_fib_indices)*list_apoptosis_coeff[1]), math.floor(len(m_cho_indices)*list_apoptosis_coeff[2]), math.floor(len(m_ost_indices)*list_apoptosis_coeff[3])])
    return np_cells, out_arr

### C.5.Proliferation

In [ ]:
def func_poli(np_cells, ind_arr, n_lattice, rate):
    # mitosis
    if n_lattice == 1:
        r = np_cells[:, ind][(np_cells[:, new_part] == 100) & (np_cells[:, in_elemnet] != -1) & (np_cells[:, lattice] == n_lattice)]
        r_len = len(r)
        random.shuffle(r)
    else:
        r = np_cells[:, ind][(np_cells[:, new_part] == 100) & (np_cells[:, in_elemnet] != -1) & (np_cells[:, lattice] == n_lattice) & (np_cells[:, stimulus] == n_lattice)]
        r_len = len(np_cells[:, ind][(np_cells[:, in_elemnet] != -1) & (np_cells[:, lattice] == n_lattice)])
        random.shuffle(r)
    
    len_max = math.floor(r_len*rate)
    len_r = len(r)
    pol_cell_count = 0
    q = 0
    
    while ((pol_cell_count < len_max) and (q < len_r)):
        
        p = int(r[q])
        q += 1
        
        x, y, z = np_cells[p, cell_x:cell_mm_x]
        x_p, x_m, y_p, y_m, z_p, z_m = np_cells[p, neighbour_p_x:]
        
        possibilities = [0, 1, 2, 3, 4, 5]
        random.shuffle(possibilities)
        
        poli = True 
        while poli and len(possibilities) > 0:
            random_index = possibilities[0]
            if (random_index == 0):
                a_x, b_y, c_z = int(x_p), int(y), int(z) 
            elif (random_index == 1):
                a_x, b_y, c_z = int(x_m), int(y), int(z) 
            elif (random_index == 2):
                a_x, b_y, c_z = int(x), int(y_p), int(z) 
            elif (random_index == 3):
                a_x, b_y, c_z = int(x), int(y_m), int(z) 
            elif (random_index == 4):
                a_x, b_y, c_z = int(x), int(y), int(z_p) 
            elif (random_index == 5):
                a_x, b_y, c_z = int(x), int(y), int(z_m) 
                
            bb = ind_arr[a_x][b_y][c_z]

            if (np_cells[bb, lattice] == 0):
                
                pol_cell_count += 1
                # change age of current cell
                np_cells[p, age] = 1
#               # add new cell
                np_cells[bb, age:in_elemnet] = 1, n_lattice
                poli = False

            else:
                possibilities = possibilities[1:]
         
    
    np_cells[:, age:in_elemnet][np_cells[:, in_elemnet] == -1] = 666, -1

    return np_cells, pol_cell_count

### C.6.Migration

In [ ]:
# @jit
def func_mig_helper(r, np_cells, ind_arr):
    jumps = 0
    for b in r:
        age_current_cell = np_cells[int(b), age]
        x, y, z = np_cells[int(b), cell_x:cell_mm_x]
        x_p, x_m, y_p, y_m, z_p, z_m = np_cells[int(b), neighbour_p_x:]
        
        possibilities = [0, 1, 2, 3, 4, 5]
        random.shuffle(possibilities)

        mitosis = True 
        while mitosis and len(possibilities) > 0:
            random_index = possibilities[0]
            if (random_index == 0):
                a_x, b_y, c_z = int(x_p), int(y), int(z) 
            elif (random_index == 1):
                a_x, b_y, c_z = int(x_m), int(y), int(z) 
            elif (random_index == 2):
                a_x, b_y, c_z = int(x), int(y_p), int(z) 
            elif (random_index == 3):
                a_x, b_y, c_z = int(x), int(y_m), int(z) 
            elif (random_index == 4):
                a_x, b_y, c_z = int(x), int(y), int(z_p) 
            elif (random_index == 5):
                a_x, b_y, c_z = int(x), int(y), int(z_m) 
                
            bb = ind_arr[a_x][b_y][c_z]

            if (np_cells[bb, lattice] == 0):
                # remove cell
                np_cells[int(b), age:in_elemnet] = 0, 0
#               # add removed cell
                np_cells[bb, age:in_elemnet] = age_current_cell, 1
                jumps += 1
                mitosis = False

            else:
                possibilities = possibilities[1:]
    return np_cells, jumps

    

In [ ]:
# @jit
def func_mig(np_cells, ind_arr, number_jumps, out_arr):
    
    # migration
    jum_count = []
    print('jumps: ', end='')
    
    for a in range(number_jumps):
        jumps = 0
        r = np_cells[:, ind][(np_cells[:, new_part] == 100) & (np_cells[:, in_elemnet] != -1) & (np_cells[:, lattice] == 1)]
        random.shuffle(r)
        
        np_cells, jumps = func_mig_helper(r, np_cells, ind_arr)
        
        jum_count.append(jumps)
        np_cells[:, age:in_elemnet][np_cells[:, in_elemnet] == -1] = 666, -1
        print(a+1, end=', ')
        
    out_arr.append(jum_count)
    print('done!')
    print(jum_count)
    return np_cells, out_arr

### C.7.MSCs Seeding

In [ ]:
def func_seeding(np_cells, ind_arr, prob):
    r = np_cells[:, ind][(np_cells[:, lattice] == 2)]
    
    for c in r:
        x, y, z = np_cells[int(c), cell_x:cell_mm_x]
        x_p, x_m, y_p, y_m, z_p, z_m = np_cells[int(c), neighbour_p_x:]
        
        for p in [x, x_p, x_m]:
            for q in [y, y_p, y_m]:
                for r in [z, z_p, z_m]:
                    
                    bb1 = ind_arr[int(p)][int(q)][int(r)]
                    rp = random.random()
                    if ((np_cells[bb1, lattice] == 0) & (rp <= prob)):
                        np_cells[bb1, age:in_elemnet] = 1
    
    return np_cells

## C.8.Scaffold Degradation

In [ ]:
def func_surf_erosion_helper(np_cells, ind_arr, index):
    x, y, z = np_cells[int(index), cell_x:cell_mm_x]
    x_p, x_m, y_p, y_m, z_p, z_m = np_cells[int(index), neighbour_p_x:]
      
    bb0 = ind_arr[int(x)][int(y)][int(z)]
    bb1 = ind_arr[int(x_p)][int(y)][int(z)]
    bb2 = ind_arr[int(x_m)][int(y)][int(z)]
    bb3 = ind_arr[int(x)][int(y_p)][int(z)]
    bb4 = ind_arr[int(x)][int(y_m)][int(z)]
    bb5 = ind_arr[int(x)][int(y)][int(z_p)]
    bb6 = ind_arr[int(x)][int(y)][int(z_m)]     
        
    l = np.unique(np.array([bb1, bb2, bb3, bb4, bb5, bb6]))
    
    final_cord= np.delete(l, np.where(l==bb0))
    
    con=len([np_cells[x, new_part] for x in final_cord if np_cells[x, new_part] == 200])
    
    return con

In [ ]:
def func_surf_erosion(on_off, np_cells, ind_arr, non_ero_degree_surface, non_ero_degree_bulk, ero_ratio_surface, ero_ratio_bulk):
    if (on_off == 1):
        n_con = [func_surf_erosion_helper(np_cells, ind_arr, i[0]) for i in np_cells]
        
        points_to_erode_surface = (np_cells[:, new_part] == 200)  & (np.array(n_con) < non_ero_degree_surface)
        points_to_erode_bulk = (np_cells[:, new_part] == 200)  & (np.array(n_con) >= non_ero_degree_bulk)
        points_to_between = (np_cells[:, new_part] == 200)  & (np.array(n_con) < non_ero_degree_bulk) & (np.array(n_con) >= non_ero_degree_surface)
        
        selected_indices_surface = np.where(points_to_erode_surface)[0]
        selected_indices_bulk = np.where(points_to_erode_bulk)[0]
        selected_indices_between = np.where(points_to_between)[0]
        
        choices_surface = np.random.choice(selected_indices_surface, size = math.floor(len(selected_indices_surface)*ero_ratio_surface), replace = False)
        choices_bulk = np.random.choice(selected_indices_bulk, size = math.floor(len(selected_indices_bulk)*ero_ratio_bulk), replace = False)

        no_erode_points_surface = len(np_cells[choices_surface])
        np_cells[:, new_part][choices_surface] = 100
        np_cells[:, age:in_elemnet][choices_surface] = 0
        
        no_erode_points_bulk = len(np_cells[choices_bulk])
        np_cells[:, new_part][choices_bulk] = 100
        np_cells[:, age:in_elemnet][choices_bulk] = 0
        
        return np_cells, no_erode_points_surface, no_erode_points_bulk, len(selected_indices_surface), len(selected_indices_bulk), len(selected_indices_between)
    else:
        return np_cells, 0

## D.Main Code

### Setting up Initial Conditions

In [ ]:
# MSCs differentiation rates [in_activity_period, after_activity_period]
msc_diff_coeff = [0.3, 0.06]

In [ ]:
# Apoptosis rates [fibroblasts, chondrocytes, mature_osteoblasts]
apoptosis_coeff = [0.05, 0.05, 0.1, 0.16]

In [ ]:
# Proliferation rates [MSCs, fibroblasts, chondrocytes, mature_osteoblasts]
# in_activity_period
poli_ratio_activity = [0.6, 0.55, 0.2, 0.3]
# after_activity_period
poli_ratio = [0.12, 0.11, 0.04, 0.06]

In [ ]:
# output csv arrays
diff_csv_arr = []
pol_csv_arr = []
mig_csv_arr = []
erode_csv_arr = []
cell_count_csv_arr = []

In [ ]:
abaqus_command = "abaqus job=NewAbaqusFileName ask_delete=OFF standard_parallel=all cpus=8 interactive"

### Setting up Initial Lattice

In [ ]:
copy_lattice_cells = np.copy(lattice_cells)

In [ ]:
# to change
copy_callus_elements = np.copy(callus_elements[:, :-1])
copy_scaffold_elements = np.copy(scaffold_elements[:, :-1])

In [ ]:
# assigning age = 0 and lattice = 0 for callus cells
copy_lattice_cells[:, age:in_elemnet][copy_lattice_cells[:, orginal_part] == 100] = 0

In [ ]:
# assigning age = 1000 and lattice = 2 for upper and lower bone celles
copy_lattice_cells[:, age][copy_lattice_cells[:, orginal_part] == 111] = 1000
copy_lattice_cells[:, lattice][copy_lattice_cells[:, orginal_part] == 111] = 2
copy_lattice_cells[:, age][copy_lattice_cells[:, orginal_part] == 112] = 1000
copy_lattice_cells[:, lattice][copy_lattice_cells[:, orginal_part] == 112] = 2

### Main Iterations

In [ ]:
# seeding MSC from top and bottom near bone with age 1 seeding in 100% areas around bone
copy_lattice_cells = func_seeding(copy_lattice_cells, map_arr_helper, 0.3)

# old code
# copy_lattice_cells[:, 7:9][(copy_lattice_cells[:, 9] != -1) & ((copy_lattice_cells[:, 5] < (-0.05)) | (copy_lattice_cells[:, 5] > (6.05)))] = 1

In [ ]:
mature_cell_age = 6

In [ ]:
msc_apoptosis_age = 7

In [ ]:
iteration_days = 90

In [ ]:
ACTIVITY_MAX = 7

In [ ]:
# Surface erosion control
# 1 --> on
# 0 --> off 
surface_erosion_on = 1
num_degree_surf=3
num_degree_bulk=6
d_ratio_surface = 0.03
d_ratio_bulk = 0.00015

In [ ]:
for j in range(iteration_days):
    
    print("Day " + str(j + 1))
    
    if (j == 0):
        prepare_simulation_file(len(copy_callus_elements), len(copy_scaffold_elements))
        print("Running Abaqus.....")
        abaqus_process = subprocess.run(abaqus_command, shell=True, capture_output=True, check=True)
        for L in abaqus_process.stdout.decode("utf-8").split("\n"):
            print(L)
        sim_elem_vals = calculate_stimulus()
        
        lookup_dict_100 = dict(zip(sim_elem_vals[:, 1][sim_elem_vals[:, 0] == 100], sim_elem_vals[:, 2][sim_elem_vals[:, 0] == 100]))
        lookup_dict_100[-1] = -1
        copy_lattice_cells[:, stimulus][copy_lattice_cells[:, orginal_part] == 100] =  [lookup_dict_100.get(element, -1) for element in copy_lattice_cells[:, in_elemnet][copy_lattice_cells[:, orginal_part] == 100]]
        
        lookup_dict_200 = dict(zip(sim_elem_vals[:, 1][sim_elem_vals[:, 0] == 200], sim_elem_vals[:, 2][sim_elem_vals[:, 0] == 200]))
        lookup_dict_200[-1] = -1
        copy_lattice_cells[:, stimulus][copy_lattice_cells[:, orginal_part] == 200] =  [lookup_dict_200.get(element, -1) for element in copy_lattice_cells[:, in_elemnet][copy_lattice_cells[:, orginal_part] == 200]]

        
    if ((j + 1) % 3 == 0):
        update_simulation_file(len(copy_callus_elements), len(copy_scaffold_elements), copy_lattice_cells)
        print("Running Abaqus.....")
        abaqus_process = subprocess.run(abaqus_command, shell=True, capture_output=True, check=True)
        for L in abaqus_process.stdout.decode("utf-8").split("\n"):
            print(L)
        sim_elem_vals = calculate_stimulus()
        
        lookup_dict_100 = dict(zip(sim_elem_vals[:, 1][sim_elem_vals[:, 0] == 100], sim_elem_vals[:, 2][sim_elem_vals[:, 0] == 100]))
        lookup_dict_100[-1] = -1
        copy_lattice_cells[:, stimulus][copy_lattice_cells[:, orginal_part] == 100] =  [lookup_dict_100.get(element, -1) for element in copy_lattice_cells[:, in_elemnet][copy_lattice_cells[:, orginal_part] == 100]]
        
        lookup_dict_200 = dict(zip(sim_elem_vals[:, 1][sim_elem_vals[:, 0] == 200], sim_elem_vals[:, 2][sim_elem_vals[:, 0] == 200]))
        lookup_dict_200[-1] = -1
        copy_lattice_cells[:, stimulus][copy_lattice_cells[:, orginal_part] == 200] =  [lookup_dict_200.get(element, -1) for element in copy_lattice_cells[:, in_elemnet][copy_lattice_cells[:, orginal_part] == 200]]

        
    # dif
    if (j > (mature_cell_age - 1)):
        print("Cell Differentiation and Apoptosis:", end = ' ')
        copy_lattice_cells, diff_csv_arr = func_diff(copy_lattice_cells, map_arr_helper, mature_cell_age, j, msc_diff_coeff, apoptosis_coeff, diff_csv_arr, msc_apoptosis_age)
        

    # pol
    print("Cell Proliferation:", end = ' ')
    lattice_values, counts = np.unique(copy_lattice_cells[:, lattice], return_counts=True)
    cell_count_dict = dict(zip(lattice_values, counts))
    list_cell_type = [1, 5, 4, 2]
    pol_arr = [j + 1]
    for ct in range(4):
        n_lattice = list_cell_type[ct]
        try:
            if (cell_count_dict[n_lattice] > 0):
                if j < ACTIVITY_MAX:
                    copy_lattice_cells, pol_count = func_poli(copy_lattice_cells, map_arr_helper, n_lattice, poli_ratio_activity[ct])
                else:
                    copy_lattice_cells, pol_count = func_poli(copy_lattice_cells, map_arr_helper, n_lattice, poli_ratio[ct])
                pol_arr.append(pol_count)
            else:
                pol_arr.append(0)
        except KeyError:
            pol_arr.append(0)
    pol_csv_arr.append(pol_arr)
    print(pol_arr)
    
    # mig
    print("Cell Migration --", end = ' ')
    
    # velocity = 30 micrometers/hour (30*24/(1/x)); x = the number of cells in lattice creation code above.
    ## change belove n_jumps to auto adjust speed
    n_jumps = 14
    copy_lattice_cells, mig_csv_arr = func_mig(copy_lattice_cells, map_arr_helper, n_jumps, mig_csv_arr)
    
    print("Scaffold Erosion:", end = ' ')
    copy_lattice_cells, erc_surface, erc_bulk, a, b, c = func_surf_erosion(surface_erosion_on, copy_lattice_cells, map_arr_helper, num_degree_surf, num_degree_bulk, d_ratio_surface, d_ratio_bulk)
    erode_csv_arr.append([(j + 1), erc_surface, erc_bulk, a, b, c, a+b+c])
    print([(j + 1), erc_surface, erc_bulk, a, b, c, a+b+c])
    
    print("Write results file")
    df_cells_new = pd.DataFrame(copy_lattice_cells[:, orginal_part:neighbour_p_x], columns =['in_part', 'in_part_new', 'x', 'y', 'z', 'x_mm', 'y_mm', 'z_mm', 'age', 'lattice', 'in_elemnet', 'stimulus'])
    result_file_name = 'raw_cells_degradation_NN_' + str(j + 1)+ '.csv'
    df_cells_new.to_csv(result_file_name)
    
    
    
    print("Increase age")
    copy_lattice_cells[:, age][(copy_lattice_cells[:, age] > 0) & (copy_lattice_cells[:, age] != 666)] += 1

    
    print("EOD Cell Count:", end = ' ')
    msc_count = len(copy_lattice_cells[(copy_lattice_cells[:, in_elemnet] != -1) & (copy_lattice_cells[:, orginal_part] != 111) & (copy_lattice_cells[:, orginal_part] != 112) & (copy_lattice_cells[:, lattice] == 1)])
    fib_count = len(copy_lattice_cells[(copy_lattice_cells[:, in_elemnet] != -1) & (copy_lattice_cells[:, orginal_part] != 111) & (copy_lattice_cells[:, orginal_part] != 112) & (copy_lattice_cells[:, lattice] == 5)])
    cho_count = len(copy_lattice_cells[(copy_lattice_cells[:, in_elemnet] != -1) & (copy_lattice_cells[:, orginal_part] != 111) & (copy_lattice_cells[:, orginal_part] != 112) & (copy_lattice_cells[:, lattice] == 4)])
    ost_count = len(copy_lattice_cells[(copy_lattice_cells[:, in_elemnet] != -1) & (copy_lattice_cells[:, orginal_part] != 111) & (copy_lattice_cells[:, orginal_part] != 112) & (copy_lattice_cells[:, lattice] == 2)])
    count_arr = [(j + 1), msc_count, fib_count, cho_count, ost_count]
    cell_count_csv_arr.append(count_arr)
    print(count_arr)
    
    
    print("*********************************************************")

In [ ]:
df_dif = pd.DataFrame(diff_csv_arr, columns = ["iteration", "diff_MSCs", "dead_MSCs", "dead_fibroblasts", "dead_chondrocytes"
                                              , "dead_osteoblasts"])
df_dif.to_csv('Differentiation_NN.csv')

In [ ]:
df_pol = pd.DataFrame(pol_csv_arr, columns = ["iteration", "MSCs", "fibroblasts", "chondrocytes", "mature_osteoblasts"])
df_pol.to_csv('Proliferation_NN.csv')

In [ ]:
df_mig = pd.DataFrame(mig_csv_arr, columns = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14])
df_mig.reset_index(inplace = True)
df_mig.rename(columns={"index": "iteration"}, inplace = True)
df_mig['iteration'] += 1
df_mig.to_csv('Migration_NN.csv')

In [ ]:
df_cell_count = pd.DataFrame(cell_count_csv_arr, columns = ["iteration", "MSCs", "fibroblasts", "chondrocytes", "mature_osteoblasts"])
df_cell_count.to_csv('cell_count_NN.csv')

In [ ]:
df_erod_count = pd.DataFrame(erode_csv_arr, columns = ["iteration", "points eroded from scaffold"])
df_erod_count.to_csv('erod_count_NN.csv')